In [0]:
SCOPE="kv-ev-scope"

STORAGE_ACCOUNT =dbutils.secrets.get(scope=SCOPE,key="source-storage-account")
CONTAINER       =dbutils.secrets.get(scope=SCOPE,key="source-container")
SAS_TOKEN       =dbutils.secrets.get(scope=SCOPE,key="source-sas-token")

spark.conf.set(
    f"fs.azure.sas.{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net",
    SAS_TOKEN
)

SOURCE_ROOT= f"wasbs://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"

print(f"Storage account : {STORAGE_ACCOUNT}")
print(f"Container       : {CONTAINER}")
print(f"Source root     : {SOURCE_ROOT}")
print("Source blob authenticated — OK")

In [0]:
SOURCE_PATH  = f"{SOURCE_ROOT}/config/"
BRONZE_VOLUME = "/Volumes/dbw_ev_intelligence_dev_7405618615455893/bronze/bronze_volume"


print(f"Source : {SOURCE_PATH}")
print(f"Bronze : {BRONZE_PATH}")


In [0]:
try:
    source_files=dbutils.fs.ls(SOURCE_PATH)
except Exception as e:
    raise Exception(f"Cannot list source config/ folder: {e}"
                    
xml_files=[f for f in source_files if f.name.endswith(".xml")]

print(f"xml files found-{len(xml_files)}")
for f in xml_files:
    print(f"{f.name<55} [{round(f.size/1024),1}KB]")

In [0]:
copied=[]
skipped=[]

for file_info in xml_files:
    dest_path=bronze_path+file_info.name
    try:
        dbutils.fs.cp(file_info.path,dest_path)
        copied.append(dest_path)
        print(f"Copied {file_info.name}")
    except Exception as e:
        skipped.append(file_info.name,str(e))
        print(f"Failed- {file_info.name}: {e})
            
print(f"\nResult: {len(copied)} copied, {len(skipped)} failed")

if skipped:
    raise Exception(f"{len(skipped)} file(s) failed — check output above.")

In [0]:
try:
    bronze_files = dbutils.fs.ls(BRONZE_PATH)
except Exception as e:
    raise Exception(f"Cannot list Bronze config/ folder: {e}")

bronze_xml = [f for f in bronze_files if f.name.endswith(".xml")]

status = "PASS" if len(bronze_xml) == len(xml_files) else "FAIL"
print(f"[{status}] Source: {len(xml_files)} files  →  Bronze: {len(bronze_xml)} files")
for f in bronze_xml:
    print(f"  {f.name}")

assert len(bronze_xml) == len(xml_files), (
    f"Count mismatch — source: {len(xml_files)}, bronze: {len(bronze_xml)}"
)
print("\nVerification passed — all XML files confirmed in Bronze Volume.")

In [0]:

if bronze_xml:
    sample = bronze_xml[0].path
    print(f"First 500 bytes of: {bronze_xml[0].name}")
    print("-" * 60)
    print(dbutils.fs.head(sample, 500))
    print("-" * 60)
    print(f"\nAll {len(bronze_xml)} XML files are in Bronze Volume and readable.")
    print("Silver layer can load these with spark.read.format('xml').option('rowTag', '').load(path)")